# Qwen2.5-VL 7B Colab Server

Google Colab에서 `Qwen/Qwen2.5-VL-7B-Instruct`를 4bit로 실행하고, ngrok URL 하나로 로컬 `K-Safety Law RAG`에서 호출할 수 있는 서버를 엽니다.

## 사용 흐름
1. Colab에서 위에서부터 셀을 순서대로 실행합니다.
2. 마지막에 ngrok 토큰만 입력합니다.
3. 출력된 `LLM_API_BASE=https://.../v1` 값을 로컬 `.env`에 넣습니다.
4. 로컬에서 사고 이미지 판단은 `/v1/accident/judgments`로 호출하고, 일반 RAG 답변은 기존 `/v1/chat/completions`로 호출합니다.

## 로컬 .env 예시
```env
LLM_PROVIDER=remote_openai
LLM_MODEL=Qwen/Qwen2.5-VL-7B-Instruct
LLM_API_BASE=https://xxxxx.ngrok-free.app/v1
LLM_API_KEY=dummy
```

In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. 의존성 설치
!pip uninstall -y transformers accelerate
!pip install -q "transformers>=4.46.0" "accelerate>=1.0.0" bitsandbytes fastapi uvicorn pyngrok nest_asyncio qwen-vl-utils pillow requests python-multipart

In [ ]:
# 3. 패키지/GPU 상태 확인
import torch
import transformers

print("transformers", transformers.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

In [ ]:
# 4. Qwen2.5-VL 7B 모델 로드
import gc
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

print("loaded", MODEL_ID)
print("device", next(model.parameters()).device)

In [ ]:
# 5. 텍스트 단독 응답 테스트
from qwen_vl_utils import process_vision_info

messages = [
    {"role": "system", "content": "한국어로 간결하게 답하세요."},
    {"role": "user", "content": "Qwen2.5-VL 7B 서버 테스트입니다. 한 문장으로 답하세요."},
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    generated_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False)

generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
print(processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip())

In [ ]:
# 6. OpenAI-compatible FastAPI 서버 실행
import base64
import json
import mimetypes
import tempfile
import threading
import traceback
import uuid
from pathlib import Path
from typing import Any

import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field
from qwen_vl_utils import process_vision_info

nest_asyncio.apply()

app = FastAPI(title="Qwen2.5-VL 7B Colab OpenAI-compatible API")

JUDGMENT_PROMPT = """
당신은 건설현장 안전사고 이미지 1차 분류 AI입니다.
입력된 사고 이미지를 보고 사진에 실제로 보이는 장면만 요약한 뒤 사고 유형을 분류하세요.
현장명, 시공사, 근로자 수, 공사금액 같은 더미 사업장 정보는 사용하지 마세요.
이미지에서 직접 확인되는 시각 단서와 사용자가 제공한 짧은 설명만 근거로 삼으세요.
보이지 않는 물체, 보호구, 높이, 사고 원인을 추측해서 만들지 마세요.
반드시 JSON만 출력하세요. 마크다운, 코드블록, 설명 문장은 출력하지 마세요.

분류 기준:
- 낙상: 같은 높이의 바닥에서 넘어짐, 미끄러짐, 걸려 넘어짐, 쓰러짐이 의심되는 장면. 작업자가 평평한 바닥에 누워 있거나 동료가 구조/확인 중이면 높이 단서가 없는 한 낙상으로 분류하세요.
- 추락: 사다리, 비계, 철골, 난간, 개구부, 굴착면 가장자리, 고소작업대 등 높은 위치 또는 아래로 떨어질 수 있는 단서가 명확히 보일 때만 분류하세요. 사람이 바닥에 누워 있다는 사실만으로는 추락이 아닙니다.
- 화재: 불, 연기, 폭발, 발화, 그을음, 화상 중심 사고 단서가 보일 때 분류하세요.
- 낙하물: 인양물, 자재, 공구 등이 떨어졌거나 떨어진 물체가 사람을 가격한 단서가 명확히 보일 때 분류하세요.
- 기타: 사고 유형을 판단할 시각 근거가 부족하거나 위 유형이 아님.

엄격 규칙:
- 추락은 높은 곳/가장자리/개구부/사다리/비계 같은 높이 단서가 없으면 선택하지 마세요.
- 화재는 불꽃/연기/폭발 단서가 없으면 선택하지 마세요.
- visible_clues와 evidence에는 사진에서 실제로 보이는 내용만 쓰세요.
- 확신이 낮으면 confidence를 0.6 이하로 두세요.

요청된 대표 유형은 낙상, 추락, 화재입니다.
다만 명백히 낙하물 사고이면 primary_type은 "기타"로 두고 secondary_type에 "낙하물"을 넣으세요.

JSON 스키마:
{
  "photo_summary": "사진에 보이는 장면을 한 문장으로 요약",
  "visible_clues": ["시각 단서 1", "시각 단서 2"],
  "primary_type": "낙상|추락|화재|기타",
  "secondary_type": "낙하물|없음|기타",
  "confidence": 0.0,
  "evidence": ["분류 근거 1", "분류 근거 2"]
}
""".strip()

class ChatRequest(BaseModel):
    model: str | None = None
    messages: list[dict[str, Any]]
    temperature: float = 0.1
    max_tokens: int = Field(default=768, alias="max_tokens")
    stream: bool = False

class AccidentJudgmentRequest(BaseModel):
    image_url: str | None = None
    image_base64: str | None = None
    manual_description: str = ""

def data_url_to_temp_file(data_url: str) -> str:
    header, _, payload = data_url.partition(",")
    if not payload:
        return data_url
    mime_type = "image/jpeg"
    if header.startswith("data:"):
        mime_type = header.split(";", 1)[0].replace("data:", "") or mime_type
    suffix = mimetypes.guess_extension(mime_type) or ".jpg"
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    tmp.write(base64.b64decode(payload))
    tmp.close()
    return tmp.name

def image_base64_to_temp_file(image_base64: str) -> str:
    if image_base64.startswith("data:"):
        return data_url_to_temp_file(image_base64)
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".jpg")
    tmp.write(base64.b64decode(image_base64))
    tmp.close()
    return tmp.name

def normalize_message_content(content: Any) -> list[dict[str, Any]]:
    if isinstance(content, str):
        return [{"type": "text", "text": content}]
    if not isinstance(content, list):
        return [{"type": "text", "text": str(content)}]

    normalized = []
    for item in content:
        item_type = item.get("type")
        if item_type in {"text", "input_text"}:
            normalized.append({"type": "text", "text": str(item.get("text", ""))})
        elif item_type in {"image_url", "input_image"}:
            image_url = item.get("image_url")
            if isinstance(image_url, dict):
                image_url = image_url.get("url", "")
            if not image_url:
                image_url = item.get("image", "")
            if isinstance(image_url, str) and image_url.startswith("data:"):
                image_url = data_url_to_temp_file(image_url)
            normalized.append({"type": "image", "image": image_url})
    return normalized or [{"type": "text", "text": ""}]

def normalize_messages(messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
    clean_messages = []
    for msg in messages:
        clean_messages.append({
            "role": msg.get("role", "user"),
            "content": normalize_message_content(msg.get("content", "")),
        })
    return clean_messages

def generate_text(messages: list[dict[str, Any]], max_tokens: int = 768, temperature: float = 0.1) -> str:
    clean_messages = normalize_messages(messages)
    text = processor.apply_chat_template(clean_messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(clean_messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    max_new_tokens = max(1, min(int(max_tokens or 768), 1536))
    temperature = float(temperature or 0.0)
    do_sample = temperature > 0

    generation_kwargs = {
        **inputs,
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
    }
    if do_sample:
        generation_kwargs["temperature"] = max(temperature, 0.01)
        generation_kwargs["top_p"] = 0.9

    with torch.inference_mode():
        generated_ids = model.generate(**generation_kwargs)

    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    return processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

def extract_json(text: str) -> dict:
    raw = text.strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    start = raw.find("{")
    end = raw.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"JSON object not found: {text[:500]}")
    return json.loads(raw[start:end + 1])

@app.get("/health")
def health():
    return {
        "ok": True,
        "model": MODEL_ID,
        "cuda": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }

@app.get("/v1/models")
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model"}]}

@app.post("/v1/chat/completions")
def chat(req: ChatRequest):
    try:
        text = generate_text(req.messages, max_tokens=req.max_tokens, temperature=req.temperature)
        completion_id = "chatcmpl-" + uuid.uuid4().hex

        if req.stream:
            def event_stream():
                chunk = {
                    "id": completion_id,
                    "object": "chat.completion.chunk",
                    "model": req.model or MODEL_ID,
                    "choices": [{"index": 0, "delta": {"content": text}, "finish_reason": None}],
                }
                yield "data: " + json.dumps(chunk, ensure_ascii=False) + "\\n\\n"
                yield "data: [DONE]\\n\\n"
            return StreamingResponse(event_stream(), media_type="text/event-stream")

        return {
            "id": completion_id,
            "object": "chat.completion",
            "model": req.model or MODEL_ID,
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "stop",
            }],
        }
    except Exception as exc:
        traceback.print_exc()
        return JSONResponse(status_code=500, content={"error": type(exc).__name__, "message": str(exc), "traceback": traceback.format_exc()})

@app.post("/v1/accident/judgments")
def accident_judgment(req: AccidentJudgmentRequest):
    try:
        content = [{"type": "text", "text": f"""
{JUDGMENT_PROMPT}

[사용자 보조 설명]
{req.manual_description if req.manual_description else '(없음)'}
""".strip()}]

        image_path = ""
        if req.image_base64:
            image_path = image_base64_to_temp_file(req.image_base64)
        elif req.image_url:
            image_path = data_url_to_temp_file(req.image_url) if req.image_url.startswith("data:") else req.image_url
        if image_path:
            content.append({"type": "image", "image": image_path})

        raw = generate_text([{"role": "user", "content": content}], max_tokens=900, temperature=0.0)
        judgment = extract_json(raw)
        return {"judgment": judgment, "raw": raw}
    except Exception as exc:
        traceback.print_exc()
        return JSONResponse(status_code=500, content={"error": type(exc).__name__, "message": str(exc), "traceback": traceback.format_exc()})

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("server started: http://localhost:8000")

In [ ]:
# 7. Colab 내부 API 테스트
import requests

payload = {
    "model": MODEL_ID,
    "messages": [{"role": "user", "content": "한국어로 한 문장만 답하세요. API 테스트입니다."}],
    "temperature": 0.1,
    "max_tokens": 128,
    "stream": False,
}

r = requests.post("http://localhost:8000/v1/chat/completions", json=payload, timeout=180)
print(r.status_code)
print(r.text[:2000])

In [ ]:
# 8. 사고 판단 API 테스트(이미지 없이 수동 설명으로 테스트)
import requests

payload = {
    "manual_description": "지하 굴착 공사구역에서 크레인이 철제 자재를 인양하던 중 자재가 아래로 떨어져 출입금지 구역에 있던 근로자를 가격한 것으로 보인다.",
}

r = requests.post("http://localhost:8000/v1/accident/judgments", json=payload, timeout=180)
print(r.status_code)
print(r.text[:3000])

In [ ]:
# 8-1. 업로드한 이미지만 판단하는 테스트 셀
import base64
import json
import mimetypes
import requests
from google.colab import files
from IPython.display import Image, display

uploaded = files.upload()
ACCIDENT_IMAGE_PATH = next(iter(uploaded.keys())) if uploaded else ""

if not ACCIDENT_IMAGE_PATH:
    raise ValueError("업로드된 사고 이미지가 없습니다.")

display(Image(filename=ACCIDENT_IMAGE_PATH, width=560))

mime_type = mimetypes.guess_type(ACCIDENT_IMAGE_PATH)[0] or "image/jpeg"
with open(ACCIDENT_IMAGE_PATH, "rb") as image_file:
    encoded_image = base64.b64encode(image_file.read()).decode("utf-8")

payload = {
    "image_base64": f"data:{mime_type};base64,{encoded_image}",
    "manual_description": "",
}

r = requests.post("http://localhost:8000/v1/accident/judgments", json=payload, timeout=240)
print(r.status_code)
data = r.json()
print(json.dumps(data.get("judgment", data), ensure_ascii=False, indent=2))

## ngrok 공개 URL 열기

ngrok 토큰은 https://dashboard.ngrok.com/get-started/your-authtoken 에서 복사합니다. 이 노트북에서 입력하는 키는 ngrok 키뿐입니다.

In [ ]:
# 9. ngrok 터널 열기
from getpass import getpass
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError

NGROK_TOKEN = ""  # 예: "2abc...". 비워두면 실행 중 입력받습니다.

if not NGROK_TOKEN.strip():
    NGROK_TOKEN = getpass("ngrok authtoken 입력: ")

ngrok.set_auth_token(NGROK_TOKEN.strip())

# 같은 Colab 런타임에서 이미 열린 터널이 있으면 새로 만들지 않고 재사용합니다.
tunnels = ngrok.get_tunnels()
if tunnels:
    public_url = tunnels[0].public_url
else:
    try:
        public_url = ngrok.connect(8000).public_url
    except PyngrokNgrokHTTPError as exc:
        message = str(exc)
        if "ERR_NGROK_334" in message or "already online" in message:
            raise RuntimeError(
                "같은 ngrok endpoint가 이미 온라인입니다. 기존 Colab 런타임/이전 노트북에서 "
                "ngrok 터널을 끄거나, ngrok dashboard의 Endpoints/Agents에서 해당 터널을 종료한 뒤 "
                "이 셀을 다시 실행하세요. 기존 endpoint가 정상 동작 중이면 새로 열 필요 없이 그 URL을 그대로 쓰면 됩니다."
            ) from exc
        raise

base_url = str(public_url).strip().strip('"')

print("PUBLIC_URL=", base_url)
print("LLM_API_BASE=", base_url + "/v1")
print("ACCIDENT_JUDGMENT_ENDPOINT=", base_url + "/v1/accident/judgments")

In [ ]:
# 10. 외부 URL 테스트
import requests

headers = {"ngrok-skip-browser-warning": "true"}

r = requests.get(base_url + "/health", headers=headers, timeout=30)
print("health", r.status_code, r.text)

r = requests.post(
    base_url + "/v1/chat/completions",
    headers=headers,
    json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": "한국어로 한 문장만 답하세요. ngrok 연결 테스트입니다."}],
        "temperature": 0.1,
        "max_tokens": 128,
        "stream": False,
    },
    timeout=180,
)
print("chat", r.status_code)
print(r.text[:2000])

## 로컬 K-Safety Law RAG 설정

`LLM_API_BASE`에는 출력된 `/v1` 주소를 넣습니다.

```env
LLM_PROVIDER=remote_openai
LLM_MODEL=Qwen/Qwen2.5-VL-7B-Instruct
LLM_API_BASE=https://xxxxx.ngrok-free.app/v1
LLM_API_KEY=dummy
```

일반 RAG 질의:

```cmd
conda activate p311_ragreport
cd /d "C:\\K-Safety Law RAG"
python cli.py
```

사고 이미지는 로컬에서 `judgement_test.py --image`로 전송합니다. Colab에는 이미지를 업로드하지 않아도 됩니다.